# Helmet detection - YOLOv8 training (Colab)
Runtime > Change runtime type > **T4 GPU** pehle select karein.

In [ ]:
!nvidia-smi
!pip install -q ultralytics

## 1. Dataset (Google Drive se)
`helmet_dataset.zip` ko apne Google Drive ke **My Drive** folder mein upload karein, phir neeche wala cell chalayein.

In [ ]:
import glob, os
from google.colab import drive
drive.mount('/content/drive')
zips = sorted(glob.glob('/content/drive/MyDrive/helmet_dataset*.zip'), key=os.path.getmtime)
assert zips, 'helmet_dataset.zip Drive ke My Drive folder mein nahi mila'
print('using newest zip:', zips[-1], f'({os.path.getsize(zips[-1])/1e6:.0f} MB)')
!rm -rf /content/data && mkdir -p /content/data
!unzip -q -o "{zips[-1]}" -d /content/data
# Colab ke liye data.yaml dobara likhein (Windows path hatana hai)
open('/content/data/data.yaml', 'w').write(
    'path: /content/data\ntrain: train/images\nval: valid/images\ntest: test/images\n'
    'nc: 2\nnames: [helmet, no_helmet]\n')
!ls /content/data
!ls /content/data/train/images | wc -l

## 2. Train
Agar Drive ke My Drive mein `best_v1.pt` (purana model) rakha hai to usi se **fine-tune** hoga (kam epochs, jaldi). Nahi hai to `yolov8s.pt` se shuru hoga (60 epochs, lamba).

In [ ]:
import os
from ultralytics import YOLO

V1 = '/content/drive/MyDrive/best_v1.pt'
if os.path.exists(V1):
    print('Fine-tuning from', V1)
    model, epochs, patience = YOLO(V1), 35, 10
else:
    print('Training from yolov8s.pt')
    model, epochs, patience = YOLO('yolov8s.pt'), 60, 15

model.train(data='/content/data/data.yaml', epochs=epochs, imgsz=640, batch=16, patience=patience,
            project='/content/runs', name='helmet_v2')
BEST = str(model.trainer.best)
print('best weights:', BEST)

## 3. Validate
Precision / recall / mAP dekhein. no_helmet class ka recall khaas dhyaan se.

In [ ]:
best = YOLO(BEST)
metrics = best.val(split='test')   # held-out test images (now includes caps/hats/hijab)
print('mAP50:', metrics.box.map50, ' mAP50-95:', metrics.box.map)
print('per-class mAP50:', dict(zip(best.names.values(), metrics.box.ap50)))
print('per-class precision:', dict(zip(best.names.values(), metrics.box.p)))
print('per-class recall:', dict(zip(best.names.values(), metrics.box.r)))

## 4. Apni test video par try karein (optional)
Video Colab mein upload karke path neeche dein.

In [ ]:
# best.predict(source='/content/test.mp4', conf=0.5, save=True)

## 5. best.pt download karein
Download hone wali file ka naam `best_v2.pt` hai. Isse `E:\helmet_detection\backend\models\` mein rakhein (purana `best.pt` abhi mat hatayein, pehle compare karenge).

In [ ]:
import shutil
from google.colab import files
shutil.copy(BEST, '/content/best_v2.pt')
files.download('/content/best_v2.pt')